# Kapitel 7 — Die Rechnung eines neuronalen Netzes verstehen · Notebook zum Ausfüllen

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch07_a_completer.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Die zu bearbeitenden Zellen sind mit **ZU ERGÄNZEN** gekennzeichnet. Führen Sie die Zellen der Reihe nach aus: Solange eine Antwort fehlt, melden die Prüfungen dies ohne Fehler. Die Lösung finden Sie in `ch07_corrige.ipynb` auf derselben Begleitseite.

**Inhalt.** Daten · Die Rechnungen der Lektionen 7.1 bis 7.3 · Lektion 7.4 (Mini-Batches und Epochen) · Optionaler Workshop (zwölf Nachrichten) · Aufgaben 7.1 bis 7.6

**Bevor Sie Jupyter öffnen:** Rechnen Sie zuerst von Hand den Vorwärtsdurchlauf von M7 (Lektion 7.1) und die Ableitung nach $w_{11}$ (Lektion 7.2). Die Zellen unten dienen dazu, diese Rechnungen zu **überprüfen**.

**Farbcode:** Indigo = Vorwärtsrechnung; Magenta gestrichelt = Weg des Gradienten; dicke Bernstein-Linie = untersuchtes Gewicht $w_{11}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farbcode des Buchs: Vorwärtsrechnung in Indigo, Gradient in Magenta (gestrichelt), untersuchtes Gewicht in dickem Bernstein
FORWARD, BACKWARD, STUDIED_WEIGHT = '#3949ab', '#c2185b', '#f9a825'


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def loss(y, p):
    # logarithmischer Verlust einer Nachricht (Gleichung 5.6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)))


def forward(P, x):
    # Vorwärtsdurchlauf (Gleichungen 7.1 und 7.2): gibt ein Dictionary z1, a1, z2, a2, z, p zurück
    z1 = P['w11'] * x[0] + P['w12'] * x[1] + P['b1']
    z2 = P['w21'] * x[0] + P['w22'] * x[1] + P['b2']
    a1, a2 = sigmoid(z1), sigmoid(z2)
    z = P['v1'] * a1 + P['v2'] * a2 + P['c']
    return {'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z': z, 'p': sigmoid(z)}


def draw(P, x, backward=False):
    # Das Schema des Buchs, vereinfacht: Indigo für den Vorwärtsdurchlauf, Magenta für den Rückweg, w11 in dickem Bernstein
    v = forward(P, x)
    pos = {'x1': (0, 3), 'x2': (0, 0), 'n1': (3, 3), 'n2': (3, 0), 's': (6, 1.5)}
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for a, b, name in (('x1', 'n1', 'w11'), ('x2', 'n1', 'w12'), ('x1', 'n2', 'w21'), ('x2', 'n2', 'w22'),
                      ('n1', 's', 'v1'), ('n2', 's', 'v2')):
        (xa, ya), (xb, yb) = pos[a], pos[b]
        if name == 'w11':
            ax.plot([xa, xb], [ya, yb], color=STUDIED_WEIGHT, linewidth=7, zorder=1)
        ax.annotate('', xy=(xb - 0.35, yb + (ya - yb) * 0.12), xytext=(xa + 0.3, ya),
                    arrowprops=dict(arrowstyle='-|>', color=FORWARD), zorder=2)
        t = 0.3 if a[0] == 'x' and b != ('n1' if a == 'x1' else 'n2') else 0.5
        ax.text(xa + t * (xb - xa), ya + t * (yb - ya) + 0.15, f"{name} = {P[name]:g}" + (' (untersuchtes Gewicht)' if name == 'w11' else ''),
                color=FORWARD, fontsize=8, ha='center', bbox=dict(facecolor='white', edgecolor='none', alpha=0.85))
    texts = {'x1': f"x1 = {x[0]}", 'x2': f"x2 = {x[1]}",
              'n1': f"N1\nz1 = {v['z1']:.3f}\na1 = {v['a1']:.4f}", 'n2': f"N2\nz2 = {v['z2']:.3f}\na2 = {v['a2']:.4f}",
              's': f"Ausgabe\nz = {v['z']:.3f}\np̂ = {v['p']:.3f}"}
    for k, (cx, cy) in pos.items():
        ax.text(cx, cy, texts[k], ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='circle', facecolor='white', edgecolor=FORWARD if k[0] != 'x' else 'grey'))
    if backward:
        for (x0, y0), (x1, y1), lab in (((6.6, 2.4), (3.5, 3.6), '② v1'), ((3.0, 3.9), (1.4, 3.4), '③ a1(1 − a1), ④ x1')):
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='-|>', color=BACKWARD, linestyle='--', connectionstyle='arc3,rad=0.3'))
            ax.text((x0 + x1) / 2, max(y0, y1) + 0.35, lab, color=BACKWARD, fontsize=8, ha='center')
        ax.text(6.6, 2.7, '① p̂ − y', color=BACKWARD, fontsize=8)
    ax.set_xlim(-0.8, 7.4); ax.set_ylim(-0.8, 4.4); ax.axis('off'); plt.show()

## Daten

Die Kennungen D7.1, D7.E1 und D5.1 sind die des Buchs und der CSV-Dateien.

In [ ]:
# Daten von Kapitel 7, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D7.1 — die neun Parameter des Netzes der Lektionen 7.1 bis 7.3
network = {'w11': 0.5, 'w12': 0.5, 'b1': -2, 'w21': 1, 'w22': -1, 'b2': 0, 'v1': 2, 'v2': -2, 'c': 0}

# D7.E1 — das Netz des Kollegen (Aufgaben 7.1 und 7.2)
colleague = {'w11': 1, 'w12': 1, 'b1': -4, 'w21': -1, 'w22': 2, 'b2': -1, 'v1': 2, 'v2': 1, 'c': -2}

# D5.1 — die zwölf Nachrichten aus Kapitel 5: Label, Links, verdächtige Wörter
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

M7 = (2, 3)   # Nachricht M7: 2 Links, 3 verdächtige Wörter, Spam (y = 1)
M3 = (4, 0)   # Newsletter M3: 4 Links, 0 verdächtige Wörter, legitim (y = 0)

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `message` | Kennung der Nachricht (M1…) | — | — |
| `label` | tatsächliches Label: Spam oder legitim | — | `etiquette` |
| `links` | Anzahl der Links in der Nachricht | Links | `liens` |
| `suspicious_words` | Anzahl der verdächtigen Wörter in der Nachricht | Wörter | `mots_suspects` |
| `network` | die neun Parameter des Netzes aus Lektion 7.1 (Gewichte, Bias, Konstante) | — | — |
| `colleague` | die neun Parameter des Netzes des Kollegen | — | — |
| `twelve` | die zwölf Nachrichten aus Kapitel 5 | — | — |

## Die Rechnungen der Lektionen, in Python

Diese Zellen wiederholen die Rechnungen der Lektionen 7.1 bis 7.3.

In [ ]:
# Lektion 7.1 — Vorwärtsdurchlauf der Nachricht M7 (Gleichungen 7.1 und 7.2)
v = forward(network, M7)
print(f"N1: z1 = {v['z1']:.3f}, a1 = {v['a1']:.4f}")
print(f"N2: z2 = {v['z2']:.3f}, a2 = {v['a2']:.4f}")
print(f"Ausgabe: z = {v['z']:.3f}, p̂ = {v['p']:.3f}, Verlust = {loss(1, v['p']):.3f}")
print(f"M3 (Newsletter): a2 = {forward(network, M3)['a2']:.3f}, p̂ = {forward(network, M3)['p']:.3f}")
draw(network, M7)

In [ ]:
# Lektion 7.2 — die Kettenregel für w11 (Gleichung 7.3), Faktor für Faktor
f1 = v['p'] - 1                  # ① Verlust ← z: p̂ − y
f2 = network['v1']                # ② z ← a1: v1
f3 = v['a1'] * (1 - v['a1'])     # ③ a1 ← z1: Ableitung der Sigmoidfunktion, mit der AKTIVIERUNG a1
f4 = M7[0]                       # ④ z1 ← w11: x1
d_w11 = f1 * f2 * f3 * f4
print(f"① {f1:.3f}  ② {f2:g}  ③ {f3:.4f}  ④ {f4}  → Ableitung = {d_w11:.3f}")

# Kontrolle durch eine kleine numerische Störung (wie die Sekanten aus Lektion 3.3)
def loss_w11(w):
    return loss(1, forward(dict(network, w11=w), M7)['p'])
h = 0.0001
print(f"numerische Ableitung: ({loss_w11(0.5 + h):.6f} − {loss_w11(0.5 - h):.6f}) / (2h) = "
      f"{(loss_w11(0.5 + h) - loss_w11(0.5 - h)) / (2 * h):.4f}")

# Eine Aktualisierung von Hand, η = 0,5, dann wird der Vorwärtsdurchlauf wiederholt
eta = 0.5
w11_new = round(0.5 - eta * d_w11, 3)   # auf Tausendstel gerundet, wie im Buch: 0,655
v_n = forward(dict(network, w11=w11_new), M7)
print(f"w11: 0,5 → {w11_new:.3f}; z1 = {v_n['z1']:.2f}, a1 = {v_n['a1']:.4f}, z = {v_n['z']:.3f}, "
      f"p̂ = {v_n['p']:.3f}, Verlust = {loss(1, v_n['p']):.3f}")
draw(network, M7, backward=True)

ws = np.linspace(-1, 3, 200)
plt.figure(figsize=(6, 3))
plt.plot(ws, [loss_w11(w) for w in ws], color='dimgrey', label='Verlust von M7 in Abhängigkeit von w11')
plt.plot(ws[25:126], loss_w11(0.5) + d_w11 * (ws[25:126] - 0.5), '--', color=BACKWARD, label='Tangente bei 0,5')
plt.scatter([0.5, w11_new], [loss_w11(0.5), loss_w11(w11_new)], color=['black', FORWARD], zorder=3)
plt.xlabel('w11'); plt.ylabel('Verlust von M7'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lektion 7.3 — Backpropagation: die neun Ableitungen (Gleichung 7.4), dann eine Aktualisierung ALLER Parameter
def backward(P, x, y):
    v = forward(P, x)
    d = v['p'] - y                                   # Fehlersignal der Ausgabe
    d1 = d * P['v1'] * v['a1'] * (1 - v['a1'])       # Fehlersignal von N1
    d2 = d * P['v2'] * v['a2'] * (1 - v['a2'])       # Fehlersignal von N2
    return {'w11': d1 * x[0], 'w12': d1 * x[1], 'b1': d1, 'w21': d2 * x[0], 'w22': d2 * x[1], 'b2': d2,
            'v1': d * v['a1'], 'v2': d * v['a2'], 'c': d}


def update(P, g, eta):
    return {k: P[k] - eta * g[k] for k in P}        # alle Ableitungen werden VOR dem Ändern berechnet


g = backward(network, M7, 1)
new = update(network, g, 0.5)
print(pd.DataFrame({'Wert': network, 'Ableitung': g, 'neuer Wert': new}).round(3))
print(f"M7: p̂ = {forward(new, M7)['p']:.3f}, Verlust = {loss(1, forward(new, M7)['p']):.3f}")
print(f"M3 vorher: p̂ = {forward(network, M3)['p']:.3f}, Verlust = {loss(0, forward(network, M3)['p']):.3f}; "
      f"nachher: p̂ = {forward(new, M3)['p']:.3f}, Verlust = {loss(0, forward(new, M3)['p']):.3f}")

## Lektion 7.4 — In kleinen Portionen lernen: Mini-Batches und Epochen

Netz im Ausgangszustand (D7.1), zwölf Nachrichten aus Kapitel 5. Damit Sie die Rechnung von Hand verfolgen können, bewegt sich nur die Ausgabekonstante $c$. Ableitung eines Batches = Mittelwert der Ableitungen seiner Nachrichten (Gleichung 7.5).

In [ ]:
# Lektion 7.4 — Summe s, Wahrscheinlichkeit und Fehlersignal δ = p̂ − y der zwölf Nachrichten, am Anfang (c = 0)
X12 = list(zip(twelve['links'], twelve['suspicious_words']))
y12 = [1 if e == 'spam' else 0 for e in twelve['label']]


def p_with_c(x, c_out):
    return forward(dict(network, c=c_out), x)['p']    # nur die Ausgabekonstante ändert sich


deltas = np.array([p_with_c(x, 0) - y_ for x, y_ in zip(X12, y12)])
print(pd.DataFrame({'message': twelve['message'], 'y': y12, 's': [forward(network, x)['z'] for x in X12],
                    'p̂': [p_with_c(x, 0) for x in X12], 'δ': deltas}).round(3).to_string(index=False))
print(f"Ableitung nach c — nur M7: {deltas[6]:.3f}; Batch M1 bis M4: {deltas[:4].mean():.3f}; "
      f"alle zwölf: Summe {deltas.sum():.3f}, Mittelwert {deltas.mean():.3f}")
print(f"Batches M5 bis M8: {deltas[4:8].mean():.3f}; M9 bis M12: {deltas[8:].mean():.3f}; "
      f"Mittelwert der drei Batches: {np.mean([deltas[:4].mean(), deltas[4:8].mean(), deltas[8:].mean()]):.3f}")


def loss_c(c_out):
    return np.mean([loss(y_, p_with_c(x, c_out)) for x, y_ in zip(X12, y12)])


# Eine Epoche: drei Aktualisierungen, jede auf ihrem Batch, jede ausgehend von dem Wert, den die vorherige hinterlassen hat
c_batch, path = 0.0, [0.0]
for k in range(3):
    batch = range(4 * k, 4 * k + 4)
    d = np.mean([p_with_c(X12[i], c_batch) - y12[i] for i in batch])
    c_batch = c_batch - 0.5 * d
    path.append(c_batch)
    print(f"Aktualisierung {k + 1} (M{4 * k + 1} bis M{4 * k + 4}): Ableitung {d:.4f} → c = {c_batch:.3f}")
c_one_step = 0 - 0.5 * deltas.mean()
print(f"mittlerer Verlust der zwölf: {loss_c(0):.3f} → {loss_c(c_batch):.3f} nach einer Epoche (3 Aktualisierungen); "
      f"ein einziger Schritt auf allen zwölf: c = {c_one_step:.3f}, Verlust {loss_c(c_one_step):.3f}")
plt.figure(figsize=(6, 2.6))
plt.plot(range(4), path, marker='o', color=FORWARD, label='eine Epoche, Batches zu je vier')
plt.plot([0, 1], [0, c_one_step], marker='s', color='black', label='ein Schritt auf allen zwölf')
plt.plot([0, 1], [0, -0.5 * deltas[6]], marker='^', color=BACKWARD, linestyle='--', label='nur M7')
plt.xlabel('Aktualisierung'); plt.ylabel('Konstante c'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Kurze Fragen (Lektion 7.4) — ZU ERGÄNZEN
updates_per_epoch = None   # ZU ERGÄNZEN: Aktualisierungen pro Epoche, zwölf Nachrichten in Batches zu je DREI
updates_ten_epochs = None  # ZU ERGÄNZEN: Aktualisierungen für zehn Epochen
c_batch_M6_M7 = None      # ZU ERGÄNZEN: neuer Wert von c nach einer Aktualisierung auf dem Batch {M6, M7}, η = 0,5

In [ ]:
# Prüfung der kurzen Fragen
if updates_per_epoch is None or updates_ten_epochs is None or c_batch_M6_M7 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die drei Zeilen.')
else:
    print(f'1. {updates_per_epoch} pro Epoche, {updates_ten_epochs} für zehn Epochen →',
          'richtig' if (updates_per_epoch, updates_ten_epochs) == (4, 40) else 'bitte prüfen')
    print(f'2. c = {c_batch_M6_M7:.3f} →', 'richtig' if abs(c_batch_M6_M7 + 0.5 * deltas[5:7].mean()) < 1e-9 else 'bitte prüfen')

## Optionaler Workshop — das Netz auf den zwölf Nachrichten trainieren

1000 Schritte Gradientenabstieg ($\eta = 0{,}5$), ausgehend von den Gewichten aus Lektion 7.1; bei jedem Schritt werden die Ableitungen der zwölf Nachrichten aus Kapitel 5 gemittelt. **Alle Messungen unten beziehen sich auf diese zwölf Nachrichten, die zum Training dienen**: Sie sagen nichts darüber, wie sich das Netz bei neuen Nachrichten verhält (Lektion 6.3).

In [ ]:
# Optionaler Workshop: Training auf den zwölf Nachrichten (gemessen auf den Trainingsdaten)
X12 = list(zip(twelve['links'], twelve['suspicious_words']))
y12 = [1 if e == 'spam' else 0 for e in twelve['label']]


def loss_12(P):
    return np.mean([loss(y, forward(P, x)['p']) for x, y in zip(X12, y12)])


def gradient_12(P):
    gs = [backward(P, x, y) for x, y in zip(X12, y12)]
    return {k: np.mean([g[k] for g in gs]) for k in P}     # Ableitung des mittleren Verlusts = Mittelwert der Ableitungen


P_trained, history = dict(network), [loss_12(network)]
for _ in range(1000):
    P_trained = update(P_trained, gradient_12(P_trained), 0.5)
    history.append(loss_12(P_trained))
print(f"mittlerer Verlust auf den zwölf Trainingsnachrichten: vorher {history[0]:.3f}, nachher {history[-1]:.3f}")
for name, x in (('M7', M7), ('M3', M3)):
    print(f"{name}: p̂ vorher {forward(network, x)['p']:.3f}, nachher {forward(P_trained, x)['p']:.3f}")
print('Fehler bei Schwelle 0,5 auf den zwölf: vorher', sum((forward(network, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)),
      ', nachher', sum((forward(P_trained, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)))
print('Gewichte nach dem Training:', {k: round(float(v), 2) for k, v in P_trained.items()})
plt.figure(figsize=(6, 3)); plt.plot(history, color=FORWARD)
plt.xlabel('Schritt'); plt.ylabel("mittlerer Verlust (zwölf Trainingsnachrichten)"); plt.grid(alpha=0.4); plt.show()

## Aufgabe 7.1 — Das Netz eines Kollegen lesen

*Begriffe: Schichten, Gewichte und Bias (Lektion 7.1, Gleichungen 7.1 und 7.2).* Daten D7.E1: das Dictionary `colleague`.

### a) bis e) Auf Papier, anhand der Abbildung im Buch

*Ihre Antworten:* …

In [ ]:
# ZU ERGÄNZEN: Anzahl der Parameter im Netz des Kollegen und gewichtete Summen einer leeren Nachricht
n_parameters = None   # ZU ERGÄNZEN: len(...)
empty = None            # ZU ERGÄNZEN: forward(colleague, (0, 0))

In [ ]:
# Kontrolle von a) und d)
if n_parameters is None or empty is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    print(f"{n_parameters} Parameter; leere Nachricht: z1 = {empty['z1']:g}, z2 = {empty['z2']:g} →",
          'richtig' if n_parameters == 9 and empty['z1'] == -4 and empty['z2'] == -1 else 'bitte prüfen')

## Aufgabe 7.2 — Eine Ausgabe von Hand berechnen

*Begriffe: Vorwärtsdurchlauf (Gleichungen 7.1 und 7.2), Sigmoidfunktion (Gleichung 5.4), Verlust (Gleichung 5.6), Entscheidungsregel (Gleichung 5.1).*

### a) bis c) Auf Papier

*Ihre Rechnungen:* …

In [ ]:
# ZU ERGÄNZEN: Wahrscheinlichkeit und Verlust von M6 (legitim, y = 0) und M7 (Spam, y = 1) im Netz des Kollegen
p_M6 = None         # ZU ERGÄNZEN: verwenden Sie forward(colleague, (2, 2))['p']
loss_M6 = None     # ZU ERGÄNZEN: verwenden Sie loss(y, p)
loss_M7 = None     # ZU ERGÄNZEN

In [ ]:
# Kontrolle von a) bis c)
if p_M6 is None or loss_M6 is None or loss_M7 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die drei Zeilen.')
else:
    print(f"M6: p̂ = {p_M6:.3f}, markiert: {p_M6 >= 0.5}, Verlust = {loss_M6:.3f} →",
          'richtig' if abs(p_M6 - 0.4332) < 5e-4 and abs(loss_M6 - 0.5677) < 5e-4 else 'bitte prüfen')
    print(f"M7: Verlust {loss_M7:.3f} (Kollege) gegenüber {loss(1, forward(network, M7)['p']):.3f} (Lektion 7.1) →",
          'richtig' if abs(loss_M7 - 0.5071) < 5e-4 else 'bitte prüfen')
    draw(colleague, (2, 2))

## Aufgabe 7.3 — Eine Aktivierung deuten

*Begriffe: Rolle eines verborgenen Neurons (Lektion 7.1).* Netz aus Lektion 7.1, zwölf Nachrichten aus Kapitel 5.

### a) bis d) Auf Papier

*Ihre Antworten:* …

In [ ]:
# ZU ERGÄNZEN: Aktivierung a2 jeder Nachricht, dann die Liste der Nachrichten, bei denen sie 0,8 übersteigt
twelve['a2'] = None    # ZU ERGÄNZEN: [forward(network, (l, m))['a2'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
lit_up = None        # ZU ERGÄNZEN

In [ ]:
# Kontrolle von a) und c)
if lit_up is None or twelve['a2'].isna().any():
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    twelve['a1'] = [forward(network, (l, m))['a1'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
    twelve['p̂'] = [forward(network, (l, m))['p'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
    print(twelve.round(3).to_string(index=False))
    print('a2 > 0,8:', lit_up, '→', 'richtig' if lit_up == ['M2', 'M3'] else 'bitte prüfen')
    print(f"M6: z = {forward(network, (2, 2))['z']}, p̂ = {forward(network, (2, 2))['p']} → markiert bei Schwelle 0,5: {forward(network, (2, 2))['p'] >= 0.5}")

## Aufgabe 7.4 — Eine Ableitung in Jupyter ergänzen

*Begriffe: Kettenregel (Gleichung 7.3), numerische Kontrolle (Lektion 7.2).* Das Gewicht $w_{22}$ verbindet die verdächtigen Wörter mit dem Neuron N2.

### a) und b) Auf Papier

*Ihre vier Faktoren und Ihre Erklärung:* …

In [ ]:
# ZU ERGÄNZEN: Ableitung des Verlusts nach w22, mit der Kettenregel
def derivative_w22(x, y):
    v = forward(network, x)
    return None   # ZU ERGÄNZEN: Produkt der vier Faktoren (p̂ − y) × v2 × a2(1 − a2) × x2

In [ ]:
# Kontrolle von c): Kettenregel gegen numerische Störung
if derivative_w22(M7, 1) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    h = 0.0001
    num = (loss(1, forward(dict(network, w22=-1 + h), M7)['p']) - loss(1, forward(dict(network, w22=-1 - h), M7)['p'])) / (2 * h)
    print(f"Kettenregel: {derivative_w22(M7, 1):.4f} →", 'richtig' if abs(derivative_w22(M7, 1) - 0.3896) < 5e-4 else 'bitte prüfen')
    print(f"numerische Ableitung: {num:.4f} → Abweichung {abs(num - derivative_w22(M7, 1)):.1e} →",
          'richtig' if abs(num - derivative_w22(M7, 1)) < 1e-6 else 'bitte prüfen')

## Aufgabe 7.5 — Vorhersagen, dann prüfen: die Wirkung einer Gewichtsänderung

*Begriffe: Ableitung und Näherung (Lektionen 3.3 und 7.2).*

### a) Ihre Vorhersage

*Vorhergesagte Verluste für $w_{11} = 0{,}6$ und $w_{11} = 2{,}5$:* …

In [ ]:
# ZU ERGÄNZEN: exakter Verlust von M7 nach einer Änderung dw nur des Gewichts w11
def loss_after(dw):
    return None   # ZU ERGÄNZEN: wiederholen Sie den Vorwärtsdurchlauf mit dict(network, w11=0.5 + dw)

In [ ]:
# Kontrolle von b)
if loss_after(0.1) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    L0 = loss(1, forward(network, M7)['p'])
    for dw in (0.1, 2):
        print(f"dw = {dw}: vorhergesagt {L0 + d_w11 * dw:.3f}, exakt {loss_after(dw):.3f}")
    print('→', 'richtig' if abs(loss_after(0.1) - 0.3716) < 5e-4 and abs(loss_after(2) - 0.2126) < 5e-4 else 'bitte prüfen')

### c) Ihre Deutung

…

## Aufgabe 7.6 — Herausforderung: mehrere Gewichte auf zwei Nachrichten aktualisieren

*Begriffe: Backpropagation (Gleichung 7.4), mittlerer Verlust (Gleichung 5.7), Gradientenabstieg (Gleichung 3.7).* Nachrichten M7 (Spam) und M3 (legitim).

### a) bis c) Auf Papier

*Ihre Rechnungen:* …

In [ ]:
# ZU ERGÄNZEN: Ableitungen des MITTLEREN VERLUSTS der beiden Nachrichten (Mittelwert der Ableitungen jeder Nachricht)
def mean_gradient(P):
    g7, g3 = backward(P, M7, 1), backward(P, M3, 0)
    return None   # ZU ERGÄNZEN: ein Dictionary {Parameter: Mittelwert der beiden Ableitungen}

In [ ]:
# Kontrolle von c), dann d): die neun Parameter, ein Schritt, dann 100 Schritte
def mean_loss(P):
    return (loss(1, forward(P, M7)['p']) + loss(0, forward(P, M3)['p'])) / 2

if mean_gradient(network) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    g = mean_gradient(network)
    output_only = dict(network, v1=network['v1'] - g['v1'], v2=network['v2'] - g['v2'], c=network['c'] - g['c'])
    print(f"Start: mittlerer Verlust {mean_loss(network):.3f}")
    print(f"c) nur v1, v2, c: M7 {loss(1, forward(output_only, M7)['p']):.3f}, "
          f"M3 {loss(0, forward(output_only, M3)['p']):.3f}, Mittelwert {mean_loss(output_only):.3f} →",
          'richtig' if abs(mean_loss(output_only) - 0.3519) < 5e-4 else 'bitte prüfen')
    nine = update(network, g, 1)
    print(f"d) die neun Parameter, ein Schritt: Mittelwert {mean_loss(nine):.3f} →",
          'richtig' if abs(mean_loss(nine) - 0.2584) < 5e-4 else 'bitte prüfen')
    P, history = dict(network), []
    for _ in range(100):
        P = update(P, mean_gradient(P), 1)
        history.append(mean_loss(P))
    print(f"   100 Schritte: p̂(M7) = {forward(P, M7)['p']:.3f}, p̂(M3) = {forward(P, M3)['p']:.3f}, mittlerer Verlust {history[-1]:.3f}")
    plt.figure(figsize=(6, 3)); plt.plot(range(1, 101), history, color=FORWARD)
    plt.xlabel('Schritt'); plt.ylabel('mittlerer Verlust (M7 und M3)'); plt.grid(alpha=0.4); plt.show()

### d) Ihr Fazit

…